In [ ]:
%run langchain_common_notebook.py

In [ ]:
embeddings = databricks_embeddings.embed_documents([
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
])

In [ ]:
import pandas as pd

docs = ["Hi there!", "Oh, hello!", "What's your name?", "My friends call me World", "Hello World!"]
pd.DataFrame(embeddings)


### Text String Embedding (PGVector.from_texts)

In [ ]:
# from_texts: loads text strings into the vector store, embedding them in the process. 
# This is the most common way to load data into a vector store.    
db = PGVector.from_texts(docs, 
                         databricks_embeddings, 
                         collection_name="test_sentences", 
                         connection_string=pgvectordb_conn,
                         use_jsonb=True)

In [ ]:
# Similarity search with scores
# You can also get similarity search without scores using `db.similarity_search()`, which returns just the documents.
scored_results = db.similarity_search_with_score("hello world", k=3)

for i, (doc, score) in enumerate(scored_results, 1):
    print(f"{i}. score={score:.6f} | text={doc.page_content}")

In [ ]:
# Retriever interface (RAG-friendly)
retriever = db.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("find messages about hello")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content}")


### Document Embeddings (PGVector.from_documents)

In [ ]:
from langchain_community.document_loaders import TextLoader

raw_documents = TextLoader('./llm_notes.txt', encoding="utf-8").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

In [ ]:
db = PGVector.from_documents(   documents,
                                databricks_embeddings,
                                collection_name="llm_notes",
                                connection_string=pgvectordb_conn,
                                use_jsonb=True,
                                pre_delete_collection=True)


In [42]:
enable_logging()
db.similarity_search("softamx", k=4)
disable_logging()

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-d30067bd-1195-4a1b-a316-3c337b726ed9', 'post_parser': <function Embeddings.create.<locals>.parser at 0x00000248199B5640>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['softamx'], 'model': 'databricks-gte-large-en', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://adb-2907158998162760.0.azuredatabricks.net//serving-endpoints/embeddings
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='adb-2907158998162760.0.azuredatabricks.net' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000002481982C910>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x

### Add New Documents

In [43]:
import uuid
from langchain_core.documents import Document

# add_documents: adds documents to the vector store, embedding them in the process.
# This is useful for adding new documents to an existing collection. 
# You can also specify your own document IDs, as shown below.
enable_logging()

ids = [str(uuid.uuid4()), str(uuid.uuid4())]
db.add_documents(
    [
        Document(
            page_content="there are cats in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
        Document(
            page_content="ducks are also found in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
    ],
    ids=ids,
)

disable_logging()


DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-fe3556c1-3ce4-4de0-bb36-f679927033fa', 'post_parser': <function Embeddings.create.<locals>.parser at 0x00000248199B5A60>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['there are cats in the pond', 'ducks are also found in the pond'], 'model': 'databricks-gte-large-en', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://adb-2907158998162760.0.azuredatabricks.net//serving-endpoints/embeddings
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='adb-2907158998162760.0.azuredatabricks.net' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x00000248199B06E0>
DEBUG:httpcore.connection:sta